In [3]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

Read in the data

In [4]:
# This assumes that train.csv and test.csv are in the same folder as this notebook
train = pd.read_csv('train.csv', index_col='ID')
test = pd.read_csv('test.csv', index_col='ID')

In [5]:
train.head()

,p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,...,p32,p33,p34,p35,p36,p37,p38,p39,p40,p50
ID,,,,,,,,,,,,,,,,,,,,,
0,95.44,98.63,102.79,99.71,96.48,93.32,88.99,84.61,83.39,82.91,...,90.57,86.88,86.27,83.42,80.25,76.95,73.69,69.15,67.07,56.62
1,104.11,104.35,104.32,104.55,103.28,99.04,96.37,97.07,96.48,92.71,...,65.94,70.90,70.07,69.43,69.45,68.24,68.39,72.00,71.67,52.60
2,109.64,109.58,110.92,108.53,104.98,102.64,107.21,107.96,114.28,111.60,...,117.49,117.19,120.93,122.84,124.58,126.50,130.17,131.77,136.22,125.94
3,104.99,103.48,104.35,104.64,104.90,101.73,100.89,104.89,105.02,105.51,...,106.08,108.09,108.14,109.10,108.68,111.60,115.23,117.05,119.02,138.97
4,108.99,102.78,99.99,98.17,94.41,95.75,96.35,98.10,98.85,94.94,...,100.85,97.59,96.23,94.31,97.42,100.85,104.61,108.91,111.31,120.89


In [13]:
# Make target concept return of stock from t=40 
TARGET = (train["p50"] - train["p40"]) / train["p40"]

In [14]:
TARGET.head()

ID
0   -0.155807
1   -0.266081
2   -0.075466
3    0.167619
4    0.086066
dtype: float64

Create a hold-out set

In [34]:
# 80-20 split
train, holdout = train_test_split(train, test_size=0.2)
# holdout is the hold-out set
# train is the remaining training data 

Train a model on the training data

In [35]:
# This is a logistic regression on the raw training data
# It is intentionally not meant to be an example of a good model
sell_train = pd.Series(train["p40"] > train["p50"]).astype(int)
X_train = train.drop('p50', axis=1)
model = LogisticRegression()
model.fit(X_train, sell_train)

c:\Users\nicho\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


Make predictions on the hold-out set

In [37]:
X_holdout = holdout.drop('p50', axis=1)
sell_holdout = model.predict(X_holdout)

Evaluate the predictions on the hold-out set

In [61]:
# To simulate the constraint, we can require that no more than 40% of the hold-out
# stocks can be sold
# The above predictions do not explicitly account for this constraint, so we may
# need to randomly subsample
def score(holdout, sell_holdout):
    # Check the constraint
    K = int(0.4 * len(sell_holdout))
    n_sell = sell_holdout.sum()
    if n_sell > K:
        # It has been violated, randomly subsample
        print(f"Subsampling {n_sell} stocks down to {K}")
        ones = np.where(sell_holdout == 1)[0]
        keep = np.random.choice(ones, size=K, replace=False)
        sell_holdout = sell_holdout * 0
        sell_holdout[keep] = 1
    p40 = holdout['p40'].values
    p50 = holdout['p50'].values
    R = float((sell_holdout * (p40 - p50)).sum())
    return R

score(holdout, sell_holdout)

Subsampling 959 stocks down to 800


3050.0

Make predictions on the test set

In [68]:
sell_test = model.predict(test)
# Note again that these predictions will not necessarily follow the constraint
# The code below checks if we are following the constraint
n_sell = sell_test.sum()
print(f"Stocks to sell: {n_sell}")
print(f"Constraint satisfied: {n_sell <= 4000}")

Stocks to sell: 4910
Constraint satisfied: False


Create submission file

In [ ]:
submission = pd.DataFrame({'ID': test.index, 'sell': sell_test})
submission.to_csv('skeleton.csv', index=False)
# The created file can be submitted to Kaggle